# 第 15 章 公共財と自然独占: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/micro-basics-programming/15-public-goods-and-natural-monopoly.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
# 自然独占（逆需要 P = a − b·Q、限界費用一定、固定費用が大きい）
demand_intercept = 100  # 逆需要の切片 a
demand_slope = 1        # 逆需要の傾き b
marginal_cost = 10      # 限界費用
fixed_cost = 800        # 固定費用（正にする）
q_max = 100             # 図の数量の範囲
p_max = 100             # 図の価格の範囲
# 公共財: 人ごとの限界便益 MB = a_i − b_i·q
benefit_intercepts = (40, 20)  # 人ごとの切片
benefit_slopes = (2, 1)        # 人ごとの傾き
public_marginal_cost = 30      # 公共財の限界費用
# 財の分類
rival = True        # 競合性があるか
excludable = False  # 排除性があるか

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_loaded_lib = sys.modules.get("lib")
if _loaded_lib is not None and (
    Path(getattr(_loaded_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in tuple(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 自然独占の価格規制

In [ ]:
from lib.chapter11.monopoly import Monopoly
from lib.chapter15.natural import (
    average_cost_pricing,
    marginal_cost_pricing,
    required_subsidy,
    unregulated_outcome,
)

demand = Monopoly(demand_intercept, demand_slope, marginal_cost)
outcomes = (
    ("限界費用価格", marginal_cost_pricing(demand, fixed_cost)),
    ("平均費用価格", average_cost_pricing(demand, fixed_cost)),
    ("規制なし", unregulated_outcome(demand, fixed_cost)),
)
for name, o in outcomes:
    print(f"{name}: 価格 {o.price:.2f}、数量 {o.quantity:.2f}、"
          f"利潤 {o.profit:.2f}、総余剰 {o.total_surplus:.2f}")
print(f"限界費用価格に必要な補助金: {required_subsidy(demand, fixed_cost):.2f}")

## 公共財

In [ ]:
from lib.chapter15.public import (
    PublicGood,
    classify_good,
    public_good_optimal_quantity,
    under_provision,
    voluntary_quantity,
)

good = PublicGood(benefit_intercepts, benefit_slopes, public_marginal_cost)
print(f"最適な供給量: {public_good_optimal_quantity(good):.4f}")
print(f"自発的な供給量: {voluntary_quantity(good):.4f}")
print(f"過少供給: {under_provision(good):.4f}")
print(f"財の分類（競合 {rival}、排除 {excludable}）: "
      f"{classify_good(rival, excludable)}")

## 図

In [ ]:
from lib.chapter15.plot import plot_natural_monopoly

plot_natural_monopoly(demand, fixed_cost, q_max, p_max)

## 試してみよう

- `fixed_cost = 1600` にすると、平均費用価格は、価格 約 34.38、数量 約 65.62 になり、
  規制なしの利潤は 425.0 に減ります。固定費用が大きいほど、平均費用価格は、
  独占の価格に近づきます
- `fixed_cost = 2000` にすると、平均費用価格は、価格 50.0、数量 50.0 です。規制なしの
  利潤は 25.0 まで減ります。限界費用価格の補助金は 2000.0 です
- `public_marginal_cost = 10` にすると、最適な供給量は約 16.67、自発的な供給量は 15.0、
  過少供給は約 1.67 です。費用が低いと、**過少供給は小さく**なります
- `benefit_intercepts = (40, 20, 30)`、`benefit_slopes = (2, 1, 1)`（3 人）にすると、
  最適な供給量は 15.0 に増えますが、自発的な供給量は 5.0 のままで、
  過少供給は 10.0 です。
  **人が増えるほど、フリーライダーの問題は、大きく**なります
- `rival = False`、`excludable = False` にすると、`public`（公共財）に分類されます